In [2]:
import pandas as pd

In [7]:
files = [
    "../data/raw/ratings.csv",
    "../data/raw/movies.csv",
    "../data/raw/tags.csv",
    "../data/raw/links.csv",
    "../data/raw/genome-tags.csv",
    "../data/raw/genome-scores.csv",
]

In [17]:
df = [pd.read_csv(file) for file in files]

## **Shape de chaque fichier**

In [20]:
for index in range(len(df)):
    print(f"{files[index][12:]} : {df[index].shape[0]} rows, {df[index].shape[1]} columns\n")

ratings.csv : 25000095 rows, 4 columns

movies.csv : 62423 rows, 3 columns

tags.csv : 1093360 rows, 4 columns

links.csv : 62423 rows, 3 columns

genome-tags.csv : 1128 rows, 2 columns

genome-scores.csv : 15584448 rows, 3 columns



## **Head de chaque fichier**

In [22]:
for file in df:
    print(file.head())

   userId  movieId  rating   timestamp
0       1      296     5.0  1147880044
1       1      306     3.5  1147868817
2       1      307     5.0  1147868828
3       1      665     5.0  1147878820
4       1      899     3.5  1147868510
   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance  
3                         Comedy|Drama|Romance  
4                                       Comedy  
   userId  movieId               tag   timestamp
0       3      260           classic  1439472355
1       3      260            sci-fi  1439472256
2       4     1732      

## **Dtypes de chaque variable**

In [25]:
for file in df:
    print(file.dtypes)

userId         int64
movieId        int64
rating       float64
timestamp      int64
dtype: object
movieId    int64
title        str
genres       str
dtype: object
userId       int64
movieId      int64
tag            str
timestamp    int64
dtype: object
movieId      int64
imdbId       int64
tmdbId     float64
dtype: object
tagId    int64
tag        str
dtype: object
movieId        int64
tagId          int64
relevance    float64
dtype: object


## **Valeurs manquantes**

In [28]:
for file in df:
    missing_values = file.isnull().sum()
    print(missing_values)

userId       0
movieId      0
rating       0
timestamp    0
dtype: int64
movieId    0
title      0
genres     0
dtype: int64
userId        0
movieId       0
tag          16
timestamp     0
dtype: int64
movieId      0
imdbId       0
tmdbId     107
dtype: int64
tagId    0
tag      0
dtype: int64
movieId      0
tagId        0
relevance    0
dtype: int64


### **Observations sur les valeurs manquantes :**

* **Fichiers intacts :** Les tables principales (`ratings.csv`, `movies.csv` et les données "genome") ne contiennent **aucune valeur manquante**. C'est idéal pour construire notre système de recommandation de base.
* **Fichier Tags (16 valeurs manquantes) :** Il manque 16 mots-clés dans la colonne `tag`. Cela signifie que des utilisateurs ont probablement validé un champ vide. 
  * *Action prévue :* Ces 16 lignes étant inutilisables en l'état, nous pourrons simplement les supprimer lors de la phase de nettoyage.
* **Fichier Links (107 valeurs manquantes) :** Il manque 107 identifiants `tmdbId`. 
  * *Impact :* Cela n'a aucun impact sur l'algorithme de recommandation en lui-même. Cela ne posera problème que si nous décidons plus tard de nous connecter à l'API de TMDB pour récupérer les affiches ou les résumés de ces 107 films spécifiques.

## **Dates minimale et maximale**

In [33]:
for index in range(len(df)):
    if "timestamp" in df[index].columns:
        min_ts = df[index]["timestamp"].min()
        max_ts = df[index]["timestamp"].max()

        date_min = pd.to_datetime(min_ts, unit="s")
        date_max = pd.to_datetime(max_ts, unit="s")

        print(f"Minimal date : {date_min}")
        print(f"Maximal date : {date_max}")

Minimal date : 1995-01-09 11:46:49
Maximal date : 2019-11-21 09:15:03
Minimal date : 2005-12-24 13:00:10
Maximal date : 2019-11-21 06:11:36


**Observation :** Les dates minimums diffèrent entre les notes et les tags. Cela indique que la fonctionnalité de "tagging" a probablement été introduite sur la plateforme plusieurs années après le système de notation.

In [ ]:
duplicated_values = df[0].duplicated(subset=["userId", "movieId", "timestamp"]).sum()
print(f"Number of duplicated values in ratings.csv : {duplicated_values}")

duplicated_vote = df[0].duplicated(subset=["userId", "movieId"]).sum()
print(f"Number of duplicated votes in ratings.csv : {duplicated_vote}")

Number of duplicated values in ratings.csv : 0
Number of duplicated votes in ratings.csv : 0


**Observation :** Il n'y a pas de valeur dupliquée dans le fichier `ratings.csv`. Aucun utilisateur n'a noté deux fois le même film.